# Deployment

> The agents plan with the world model while their messages go through the channel: stable-marl's `World` and
> `CommunicationWrapper`, comm-core's link with C3-WM's power control, and the world model as the planners' cost.

In [ ]:
#| default_exp evaluators.deploy

In [ ]:
#| hide
from nbdev.showdoc import *

What the world model was trained for (`trainers.wm_tune`), now in the environment:

* **the messages** (`C3WMCommunication`): at every step each agent sends its view's 49 VQ tokens and its pose (52
  integers, 8 bits each: 416 bits) to the others, over comm-core's network (stable-marl's `CommunicationWrapper`);
  a message arrives or not, and each agent keeps every other agent's latest message, its age and the sender's pose
  when sent relative to where the receiver is now: what `models.jepa.latest_messages` gives the predictor in training (checked below);
* **the link** (`c3wm_link`): built for each episode's layout, as `comm.power_control.c3wm_link`'s blocks: the
  layout's ray-traced large-scale gain with Rician fading drawn round by round (`OnlineRicianFading`, as the data's),
  the power control ('perfect CSI' channel inversion, or the 'learned' one with its κ and τ), the schedule (τ),
  uncoded BPSK and a CRC (a message with a wrong bit is dropped: an erasure, as in training); or a 'perfect channel';
* **the planning** (`C3WMCost`): stable-marl's `WorldModelPolicy` with a solver in `per_agent` mode, each agent
  rolling its candidate plans out with the world model from its own frames, actions and received messages, scored
  by `JEPA.criterion`;
* **the episodes** (`deploy`): the test layouts' episodes (`utils.env_utils.LayoutWrapper`: the same goal and starts
  for the same seed), the same ones for every method: their success, length and, over a link, its reliability and power.

In [ ]:
#| export
from __future__ import annotations

import math
from typing import Callable

import gymnasium as gym
import numpy as np
import torch
from gymnasium import spaces
from torch import nn

from comm_core import Fading
from stable_marl.comm.core import AgentNetwork, node_id
from stable_marl.comm.wrapper import CommunicationWrapper

from c3jepa_wm.data.datasets import _relative_pose
from c3jepa_wm.comm.channel_data import _pair

MESSAGE_TOKENS = 49                                                 # the sender's 7 x 7 view, one VQ token per tile
POSE = 3                                                            # and its pose: x, y, heading

## The channel

In [ ]:
#| export
class OnlineRicianFading(Fading):
    """The channel between the nodes' cells, drawn round by round: a layout's large-scale gain (its `PairCache`, traced
    when first needed) times Rician small-scale fading |g|², as the data's (`comm.channel_data._fading`): drawn again
    when either node moved, drifting with `rho` while both stand still, the direct path's share K / (K + 1) from the
    cache. Both directions of a pair share its fading; `rician=False`: the large-scale gain alone."""

    def __init__(
        self,
        cache, # The layout's `PairCache`
        rho: float = 0.9, # The fading's correlation from one round to the next while both nodes stand still
        rician: bool = True, # Small-scale fading on top of the large-scale gain
    ):
        self.cache, self.rho, self.rician = cache, rho, rician
        self.node_positions, self.round, self.state = {}, 0, {}

    def reset(self, seed=None):
        super().reset(seed)
        self.node_positions, self.round, self.state = {}, 0, {}

    def update(self, time):
        self.round += 1

    def set_positions(self, positions):
        "Where the nodes are now: {node: (x, y)}."
        self.node_positions.update({k: tuple(int(v) for v in p) for k, p in positions.items()})

    def _draw(self) -> complex:
        return complex(self.rng.standard_normal(), self.rng.standard_normal()) / math.sqrt(2)

    def fade(self, a_node, b_node) -> float:
        "|g|² of the pair this round."
        first, second = sorted((a_node, b_node))
        cells = (self.node_positions[first], self.node_positions[second])
        s = self.state.get((first, second))
        if s is None or s['cells'] != cells:                         # a move: many wavelengths, a new draw
            s = self.state[(first, second)] = dict(cells=cells, w=self._draw(), phase=self.rng.uniform(0, 2 * math.pi),
                                                    round=self.round)
        elif s['round'] < self.round:                                # standing still: the scattered part drifts
            s['w'] = self.rho * s['w'] + math.sqrt(1 - self.rho ** 2) * self._draw()
            s['round'] = self.round
        k = _pair(*cells)
        d = min(self.cache.D.get(k, 0.0) / max(self.cache.P[k], 1e-30), 1.0)
        return abs(math.sqrt(d) * complex(math.cos(s['phase']), math.sin(s['phase'])) + math.sqrt(1 - d) * s['w']) ** 2

    def coefficient(self, transmission, link) -> complex:
        a, b = self.node_positions[transmission.sender], self.node_positions[transmission.receiver]
        if a == b:
            return math.inf
        gain = self.cache.gain(a, b, 'large_scale')                  # traced now if it was not
        if self.rician:
            gain *= self.fade(transmission.sender, transmission.receiver)
        return complex(math.sqrt(gain))

In [ ]:
#| hide
import types
from c3jepa_wm.comm.channel_data import _pair

class _Cache:                                                       # a stand-in PairCache: -60 dB, half of it the direct path
    def __init__(self): self.P, self.D = {}, {}
    def gain(self, a, b, channel):
        k = _pair(a, b); self.P[k], self.D[k] = 1e-6, 0.5e-6
        return 1e-6

f = OnlineRicianFading(_Cache(), rho=0.9)
f.reset(0)
tx = lambda s, r: types.SimpleNamespace(sender=s, receiver=r)
moving = []
for t in range(4000):                                               # a move every round: a new draw each, of mean 1
    f.set_positions({'a': (1, 1 + t % 2), 'b': (5, 5)}); f.update(t)
    moving.append(abs(f.coefficient(tx('a', 'b'), None)) ** 2 / 1e-6)
assert abs(np.mean(moving) - 1) < 0.05 and abs(np.corrcoef(moving[:-1], moving[1:])[0, 1]) < 0.1
f.reset(1)
still = []
for t in range(4000):                                               # standing still: the same in both directions, drifting
    f.set_positions({'a': (1, 1), 'b': (5, 5)}); f.update(t)
    h = f.coefficient(tx('a', 'b'), None)
    assert h == f.coefficient(tx('b', 'a'), None)
    still.append(abs(h) ** 2 / 1e-6)
assert np.corrcoef(still[:-1], still[1:])[0, 1] > 0.5 and abs(np.mean(still) - 1) < 0.15
f.set_positions({'b': (1, 1)})
assert f.coefficient(tx('a', 'b'), None) == math.inf                # the same cell: a perfect link

## The messages

In [ ]:
#| export
class C3WMCommunication(CommunicationWrapper):
    """stable-marl's `CommunicationWrapper` with C3-WM's messages: each agent sends its view's VQ tokens and its pose
    (`tokens`), over a link built for each episode's layout (`link`), and keeps every other agent's latest message it
    received as the world model reads them (`models.jepa.latest_messages`): `msg_tokens` (A, 49), `msg_received` (A,),
    `msg_age` (A,) and `msg_pose` (A, 3), the sender's pose when it sent it relative to the receiver's pose now (one
    row per agent; the receiver's own row never received). Each round's transmissions are kept in `records` (gain, power,
    sent, arrived, wrong tokens) when the link reports them."""

    def __init__(
        self,
        env: gym.Env, # A stable-marl MultiGrid env (e.g. with `utils.env_utils.LayoutWrapper`: the layout in its infos)
        tokens: Callable | None, # tokens(pixels (A, H, W, C) uint8) -> (A, 49) VQ tokens; None: no communication
        link: Callable | None = None, # link(layout, env) -> `AgentNetwork`'s channel / blocks / backend options; None: a perfect channel
        max_age: int = 8, # A message is kept this many rounds after it was sent (the predictor's `max_age`)
    ):
        super().__init__(env, message_fn=self._payload, message_dim=MESSAGE_TOKENS + POSE)
        self.tokens, self.link, self.max_age = tokens, link, max_age
        self.records: list = []
        A = self.num_agents
        extra = {'msg_tokens': spaces.Box(0, np.iinfo(np.int64).max, (A, MESSAGE_TOKENS), np.int64),
                 'msg_received': spaces.MultiBinary(A),
                 'msg_age': spaces.Box(0, max_age, (A,), np.int64),
                 'msg_pose': spaces.Box(-np.inf, np.inf, (A, POSE), np.int64)}
        self.observation_space = spaces.Dict({a: spaces.Dict({**self.observation_space[a].spaces, **extra})
                                              for a in range(A)})
        self._payloads = {}
        self._clear()

    def _clear(self):
        A = self.num_agents
        self._memory = dict(tokens=np.zeros((A, A, MESSAGE_TOKENS), np.int64), received=np.zeros((A, A), bool),
                            age=np.zeros((A, A), np.int64), sender=np.zeros((A, A, POSE), np.int64))

    def _payload(self, env, agent, obs):
        return self._payloads.get(agent)

    def reset(self, *, seed: int | None = None, options: dict | None = None):
        "Reset the env, build the link for its layout, and the first round of messages."
        obs, infos = self.env.reset(seed=seed, options=options)
        layout = int(np.asarray(infos[0].get('layout', -1)))
        options = self.link(layout, self.env.unwrapped) if self.link is not None else {}
        self.comm = AgentNetwork(self.num_agents, 'full_mesh', bits_per_element=8, **options)
        self.comm.reset(seed)
        self._clear()
        return self._communicate(obs, infos)

    def _communicate(self, obs: dict, infos: dict):
        A = self.num_agents
        if self.tokens is None:
            self._payloads = {}
        else:
            toks = np.asarray(self.tokens(np.stack([obs[a]['pixels'] for a in range(A)])))
            self._payloads = {a: np.r_[toks[a], np.asarray(obs[a]['position']).reshape(-1), int(obs[a]['direction'])]
                              .astype(np.int64) for a in range(A)}
        obs, infos = super()._communicate(obs, infos)
        self._record()
        return self._remember(obs), infos

    def _record(self):
        for r in self.comm.last_results:
            m = r.metadata or {}
            if 'channel_gain_db' in m and 'tx_power' in m:
                self.records.append((10 ** (m['channel_gain_db'] / 10), m['tx_power'], not m.get('outage', False),
                                     bool(r.success), m.get('wrong_tokens')))

    def _remember(self, obs: dict) -> dict:
        "Each receiver's latest message from every sender: a new one at age 0, the others a round older, expired after `max_age`."
        A, mem = self.num_agents, self._memory
        mem['age'] += 1
        for a in range(A):
            arrived = np.flatnonzero(obs[a]['message_mask'])
            for j in arrived:
                m = np.rint(obs[a]['messages'][j]).astype(np.int64)
                mem['tokens'][a, j] = m[:MESSAGE_TOKENS]
                mem['sender'][a, j] = m[MESSAGE_TOKENS:MESSAGE_TOKENS + POSE]   # its pose when it sent: x, y, direction
                mem['age'][a, j], mem['received'][a, j] = 0, True
        mem['received'] &= mem['age'] <= self.max_age
        mem['age'] = np.where(mem['received'], mem['age'], 0)
        out = dict(obs)
        for a in range(A):                                          # each message placed from where the receiver is now
            pose = _relative_pose(mem['sender'][a, :, :2], mem['sender'][a, :, 2], np.asarray(obs[a]['position']),
                                  int(obs[a]['direction']))
            out[a] = {**obs[a], 'msg_tokens': mem['tokens'][a].copy(), 'msg_received': mem['received'][a].copy(),
                      'msg_age': mem['age'][a].copy(), 'msg_pose': pose.astype(np.int64)}
        return out

def vq_tokens(vqvae, device="cpu") -> Callable:
    "`C3WMCommunication`'s `tokens`: the VQ-VAE's 49 tokens of each agent's view (its training transform: [-1, 1])."
    @torch.no_grad()
    def tokens(pixels):
        x = torch.as_tensor(pixels, device=device).permute(0, 3, 1, 2).float() / 127.5 - 1
        return vqvae.get_message_indices(x).flatten(1).cpu().numpy()
    return tokens

In [ ]:
#| hide
import stable_marl as sm
from omegaconf import OmegaConf
from comm_core import ErasureChannel
from stable_marl.policy import RandomPolicy
from c3jepa_wm.utils.env_utils import LayoutWrapper
from c3jepa_wm.models.jepa import latest_messages
from c3jepa_wm.data.datasets import _relative_pose

ENV = OmegaConf.to_container(OmegaConf.load('../configs/comm/env/findgoal.yaml'))
WORLD = {k: v for k, v in ENV.items() if k not in ('_target_', 'variations')}

calls = []
def fake_tokens(pixels):                                            # the message of the n-th round: all n (mod 256)
    calls.append(len(calls))
    return np.full((len(pixels), 49), len(calls) % 256)

world = sm.World('MultiGrid-FindGoal-15x15-v0', num_envs=1, pre_wrappers=[
    lambda e: LayoutWrapper(e, range(230, 232)),
    lambda e: C3WMCommunication(e, fake_tokens, lambda layout, env: {'channel': ErasureChannel(0.5)}, max_age=2)], **WORLD)
world.set_policy(RandomPolicy(seed=0))
world.reset(seed=[0])
snapshot = lambda infos: {k: np.copy(v) for k, v in infos.items() if isinstance(v, np.ndarray)}   # the pool reuses its arrays
steps = [snapshot(world.infos)]
for _ in range(12):
    _, _, _, _, world.infos = world.envs.step(world.policy.get_action(world.infos), mask=np.ones(1, bool))
    steps.append(snapshot(world.infos))
get = lambda k: np.stack([s[k][0, 0] for s in steps])               # (S, ...) of the one env
messages, mask = np.rint(get('messages')).astype(np.int64), get('message_mask')    # (S, receiver, sender, 52), (S, A, A)
position, direction = get('position'), get('direction')
assert 0.2 < mask[:, 0, 1].mean() < 0.8                            # about half of them arrived
for a in range(2):                                                  # each receiver's memory: latest_messages' exactly
    receiver = np.concatenate([position[:, a], direction[:, a][:, None]], -1)
    m = latest_messages(torch.as_tensor(messages[None, :, a, :, :49]), torch.as_tensor(mask[None, :, a]),
                        torch.as_tensor(messages[None, :, a, :, 49:52]), torch.as_tensor(receiver[None]),
                        steps=len(steps), max_age=2)
    assert (get('msg_received')[:, a] == m['received'][0].numpy()).all()
    assert (get('msg_age')[:, a] == m['age'][0].numpy()).all()
    ok = m['received'][0].numpy()
    assert (get('msg_tokens')[:, a][ok] == m['tokens'][0].numpy()[ok]).all()
    assert (get('msg_pose')[:, a][ok] == m['pose'][0].numpy()[ok]).all()
assert not get('msg_received')[:, [0, 1], [0, 1]].any()             # no one hears itself
world.close()

## The link

In [ ]:
#| export
def c3wm_link(method: str, cache_for: Callable, saved: dict | None = None, crc: bool = True, rho: float = 0.9,
              rician: bool = True, policies: dict | None = None, policy: str | None = None, target: float = 0.95) -> Callable:
    """`C3WMCommunication`'s `link` for a method: 'perfect channel' (every message arrives), or C3-WM's link
    (`comm.power_control.c3wm_link`: the layout's channel, `OnlineRicianFading`, the power, uncoded BPSK, a CRC if
    `crc`) with 'perfect CSI' channel inversion, the 'learned' power control (`saved`: `load_power_control`'s, with
    its κ and τ), or the 'lambda-policy' (Proposition 1's policy: `policies`, `comm.evaluation.load_lambda_policies`'s,
    its `policy` at `target`; it withholds by itself, τ = ∞). `cache_for(layout)` -> the layout's `PairCache`."""
    from comm_core import ChannelInversion
    from c3jepa_wm.comm.power_control import LearnedChannelInversion, c3wm_link as blocks
    from c3jepa_wm.comm.evaluation import DistributionalChannelInversion
    if method == 'perfect channel':
        return lambda layout, env: {}
    if method not in ('perfect CSI', 'learned', 'lambda-policy'):
        raise ValueError(f"method: 'perfect channel', 'perfect CSI', 'learned' or 'lambda-policy', not {method!r}")
    if method == 'lambda-policy' and (policies is None or policy not in policies['policies']):
        raise ValueError(f"'lambda-policy' needs `policies` with {policy!r}: {list((policies or {}).get('policies', {}))}")
    b = policies['budget'] if method == 'lambda-policy' else saved['budget']

    def link(layout, env):
        cache = cache_for(layout)
        fading = OnlineRicianFading(cache, rho, rician)
        heads = lambda: {node_id(i): int(agent.state.dir) for i, agent in enumerate(env.agents)}
        if method == 'perfect CSI':
            control, tau = ChannelInversion(b.snr_target_db), 0.0
        elif method == 'lambda-policy':
            control = DistributionalChannelInversion.from_saved(policies, policy, target, cache.walls,
                                                                lambda: fading.node_positions, heads)
            tau = math.inf
        else:
            control = LearnedChannelInversion.from_saved(saved, cache.walls, lambda: fading.node_positions, heads)
            tau = saved['calibration']['tau_db']
        return dict(blocks=blocks(fading, control, b.max_power_w, tau, crc=crc), noise_power=b.noise_w)
    return link

## The planning

In [ ]:
#| export
class ActionSubset(gym.ActionWrapper):
    """Each agent acts with the env's first `n` actions only, as they are (MultiGrid's 0 turn left, 1 turn right,
    2 forward; FindGoal's fourth, `pickup`, does nothing there): what the planners choose from."""

    def __init__(self, env: gym.Env, n: int = 3):
        super().__init__(env)
        self.action_space = spaces.Dict({a: spaces.Discrete(n) for a in env.action_space.spaces})

    def action(self, action):
        return action

IMAGENET = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

def jepa_transform(images: torch.Tensor) -> torch.Tensor:
    "The world model's frames (as its training's: [0, 1], then ImageNet's normalization), from (N, C, H, W) uint8."
    mean, std = (torch.tensor(v, device=images.device).view(1, 3, 1, 1) for v in IMAGENET)
    return (images.float() / 255 - mean) / std

class C3WMCost(nn.Module):
    """C3-WM's world model as a planning cost with one cost per agent (a solver's `mode='per_agent'`): each agent rolls
    the candidate plans out from its own frames, actions and received messages with the shared model (`JEPA.rollout`),
    scored by `JEPA.criterion` (the closest imagined step's distance to its goal's embedding). `messages=False`: the
    blind model's planning (no messages read)."""

    def __init__(self, model, history_size: int = 3, messages: bool = True):
        super().__init__()
        self.model, self.history_size, self.messages = model, history_size, messages

    def criterion(self, info_dict: dict, action_candidates: torch.Tensor) -> torch.Tensor:
        return self.get_cost(info_dict, action_candidates)

    @torch.no_grad()
    def get_cost(self, info_dict: dict, action_candidates: torch.Tensor) -> torch.Tensor:
        """
        info_dict: pixels (B, S, T, A, C, H, W), goal (B, S, 1, A, C, H, W), action_history (B, S, T - 1, A, 1),
                   msg_tokens / msg_received / msg_age / msg_pose (B, S, T, A, A, ...) — the policy's history
        action_candidates: (B, S, horizon, A, K) one-hot
        returns: (B, S, A)
        """
        B, S, _, A = action_candidates.shape[:4]
        plan = action_candidates.argmax(-1)                         # (B, S, horizon, A)
        pixels, history = info_dict['pixels'], info_dict.get('action_history')
        T = pixels.shape[2]
        goal = info_dict['goal'].reshape(B, S, -1, A, *pixels.shape[-3:])[:, :, -1]   # (B, S, A, C, H, W)
        costs = []
        for a in range(A):
            actions = plan[:, :, :, a]
            if history is not None and T > 1:
                past = torch.nan_to_num(history[:, :, :, a].reshape(B, S, -1).float(), nan=0.0).long()
                actions = torch.cat([past[:, :, -(T - 1):], actions], dim=2)
            info = {'pixels': pixels[:, :, :, a]}
            if self.messages and 'msg_tokens' in info_dict:
                info['messages'] = {'tokens': info_dict['msg_tokens'][:, :, :, a].long(),
                                    'received': info_dict['msg_received'][:, :, :, a].bool(),
                                    'age': info_dict['msg_age'][:, :, :, a].long(),
                                    'pose': info_dict['msg_pose'][:, :, :, a].long()}
            goal_emb = self.model.encode({'pixels': goal[:, 0, a].unsqueeze(1)})['emb']   # (B, 1, D): the same for every candidate
            info['goal_emb'] = goal_emb.unsqueeze(1).expand(B, S, 1, -1)
            info = self.model.rollout(info, actions, history_size=self.history_size)
            costs.append(self.model.criterion(info))
        return torch.stack(costs, dim=-1)

In [ ]:
#| hide
import hydra
from hydra import compose, initialize_config_dir
from pathlib import Path
torch.manual_seed(0)
with initialize_config_dir(config_dir=str(Path('../configs/train').resolve()), version_base=None):
    cfg = compose('config', overrides=['pipeline=wm_finetune', 'pipeline.model.jepa.predictor.depth=2',
                                       'pipeline.model.jepa.predictor.mlp_dim=256', 'pipeline.model.jepa.checkpoint_path=',
                                       'pipeline.model.vqvae.checkpoint_path='])
jepa = hydra.utils.instantiate(cfg.pipeline.model.jepa).eval()
for block in jepa.predictor.transformer.layers:                     # as if trained: the messages are read
    torch.nn.init.normal_(block.cross_attn.out_proj.weight, std=0.1)

B, S, T, A, H = 1, 4, 2, 2, 3
info = {'pixels': torch.randn(B, S, T, A, 3, 224, 224), 'goal': torch.randn(B, S, 1, A, 3, 224, 224),
        'action_history': torch.randint(0, 4, (B, S, T - 1, A, 1)).float(),
        'msg_tokens': torch.randint(0, 256, (B, S, T, A, A, 49)), 'msg_received': torch.ones(B, S, T, A, A, dtype=torch.bool),
        'msg_age': torch.zeros(B, S, T, A, A, dtype=torch.long), 'msg_pose': torch.zeros(B, S, T, A, A, 3, dtype=torch.long)}
info['pixels'][:, 1:] = info['pixels'][:, :1]; info['goal'][:, 1:] = info['goal'][:, :1]   # one context, S candidates
plans = torch.nn.functional.one_hot(torch.randint(0, 4, (B, S, H, A)), 4).float()
heard, blind = C3WMCost(jepa, messages=True), C3WMCost(jepa, messages=False)
cost = heard.get_cost(dict(info), plans)
assert cost.shape == (B, S, A) and torch.isfinite(cost).all()
other = dict(info, msg_tokens=(info['msg_tokens'] + 1) % 256)
assert not torch.allclose(heard.get_cost(dict(other), plans), cost)            # the messages count
assert torch.allclose(blind.get_cost(dict(other), plans), blind.get_cost(dict(info), plans))   # not for the blind one

## Deployment

In [ ]:
#| export
def find_wrapper(env, cls):
    "The wrapper of type `cls` in `env`'s chain (None if there is none)."
    while env is not None:
        if isinstance(env, cls):
            return env
        env = getattr(env, 'env', None)
    return None

def _merged(parts: list) -> dict:
    "`World.evaluate`'s results of consecutive rounds as one."
    out = {k: np.concatenate([p[k] for p in parts]) for k in ('episode_successes', 'episode_returns', 'episode_lengths', 'seeds')}
    out['success_rate'] = float(out['episode_successes'].mean()) * 100.0
    out['mean_return'] = float(out['episode_returns'].sum(axis=1).mean())
    out['mean_length'] = float(out['episode_lengths'].mean())
    return out

def deploy(
    make_world: Callable, # make_world(method) -> a stable-marl `World` (its envs wrapped in `C3WMCommunication`)
    make_policy: Callable, # make_policy(method) -> the policy (e.g. a `WorldModelPolicy` over `C3WMCost`)
    methods, # The methods' names
    episodes: int, # Episodes per method (the same ones for each: the same seeds)
    seed: int = 0, # The first episode's seed
    budget=None, # `comm.power_control.Budget`: the link's metrics (`comm.evaluation.network_metrics`), for the methods with a link
) -> dict:
    """Per method: the episodes' success rate, return and length (`World.evaluate`) and, for the methods sending over a
    link, its metrics: reached, power (× perfect CSI's), withheld, intact (every token right), wrong tokens, garbled.
    The episodes are played in rounds of the world's `num_envs`, all starting together (`reset_mode='wait'`): the envs
    planned for at once then always have histories of the same length (an env starting while others are midway would
    have its history padded with copies of its first frame, which the world model never saw)."""
    from c3jepa_wm.comm.evaluation import network_metrics
    out = {}
    for method in methods:
        world = make_world(method)
        world.set_policy(make_policy(method))
        n = world.num_envs
        parts = [world.evaluate(episodes=min(n, episodes - start), seed=seed + start, reset_mode='wait')
                 for start in range(0, episodes, n)]
        result = _merged(parts)
        rows = [r for env in world.envs.envs for r in (find_wrapper(env, C3WMCommunication).records or [])]
        result['link'] = network_metrics(rows, budget, MESSAGE_TOKENS, [0.0]) if rows and budget is not None else None
        world.close()
        out[method] = result
    return out

In [ ]:
#| hide
# end to end on two test layouts, 4 steps: the learned link over a stand-in tracer and a stand-in power-control model
import math, tempfile
from stable_marl.policy import WorldModelPolicy, PlanConfig
from stable_marl.planning.solver import CategoricalCEMSolver
from c3jepa_wm.planners.mpc import DiscreteCEMSolver
from c3jepa_wm.comm.channel_data import PairCache, wall_geometry
from c3jepa_wm.comm.power_control import Budget

tmp = Path(tempfile.mkdtemp())
caches = {}
def cache_for(layout):
    if layout not in caches:
        c = PairCache(tmp / f'pairs_{layout}.npz', ENV, layout)
        def tracer(tx, rxs, walls=c.walls):                         # free space at 2.4 GHz, 6 dB a wall, 30% direct
            out = []
            for rx in rxs:
                d, w = wall_geometry(walls, tx, rx)
                p = (0.125 / (4 * math.pi * max(d, 0.5))) ** 2 * 10 ** (-0.6 * w)
                out.append((complex(math.sqrt(p)), p, 0.3 * p))
            return out
        c.tracer = tracer
        caches[layout] = c
    return caches[layout]

class Flat:                                                         # a stand-in gain predictor: -60 dB everywhere
    steps = False
    def predict(self, X): return np.full(len(X), -60.0)
budget = Budget(snr_target_db=10, noise_dbm=-94, max_power_dbm=-10)
saved = dict(model=Flat(), history=4, budget=budget, calibration=dict(margin_db=3.0, tau_db=10.0))
class FlatDistribution:                                             # a stand-in predicted distribution: -60 dB ± 3
    levels = np.array([0.05, 0.5, 0.95])
    def quantiles(self, data): return np.tile([-57.0, -60.0, -63.0], (len(data['X_possible']), 1))
    def cdf(self, data, g_db): return np.full(len(data['X_possible']), float(g_db >= -60))
policies = dict(policies={'mean + spread, λ-policy': dict(dist=FlatDistribution(), lam={0.95: 50 * budget.noise_w})},
                history=4, budget=budget)

def make_world(method):
    tokens = None if method == 'no communication' else (lambda p: np.random.randint(0, 256, (len(p), 49)))
    link = None if method == 'no communication' else c3wm_link(method, cache_for, saved, policies=policies,
                                                               policy='mean + spread, λ-policy', target=0.95)
    return sm.World('MultiGrid-FindGoal-15x15-v0', num_envs=2, goal_conditioned=True, **{**WORLD, 'max_steps': 4},
                    pre_wrappers=[lambda e: LayoutWrapper(e, range(230, 232)), lambda e: C3WMCommunication(e, tokens, link),
                                  ActionSubset])
def make_policy(method):
    cost = C3WMCost(jepa, messages=method != 'no communication')
    solver = (DiscreteCEMSolver(cost, pop_size=4, topk=2, opt_steps=1) if method == 'learned' else   # the c3jepa planner
              CategoricalCEMSolver(cost, batch_size=None, num_samples=4, n_steps=1, topk=2, mode='per_agent'))
    return WorldModelPolicy(solver, PlanConfig(horizon=2, receding_horizon=1, history_len=3),
                            transform={'pixels': jepa_transform, 'goal': jepa_transform},
                            history_keys=('pixels', 'msg_tokens', 'msg_received', 'msg_age', 'msg_pose'))

lengths = []                                                        # the histories planned from together: one length
def checked(method):
    policy = make_policy(method)
    plan = policy.get_action
    def get_action(obs, **kw):
        out = plan(obs, **kw)
        if policy._history_buffer is not None:
            lengths.append(set(policy._history_buffer.num_strided()))
        return out
    policy.get_action = get_action
    return policy

results = deploy(make_world, checked, ['no communication', 'perfect CSI', 'learned', 'lambda-policy'], episodes=3, budget=budget)
assert results['lambda-policy']['link']['transmissions'] > 0
assert results['no communication']['link'] is None and results['learned']['link']['transmissions'] > 0
assert results['perfect CSI']['link']['reached'] == 1.0 and abs(results['perfect CSI']['link']['power'] - 1) < 1e-9
assert all(r['seeds'].tolist() == [0, 1, 2] for r in results.values())   # 3 episodes in rounds of 2: the same seeds
assert lengths and all(len(l) == 1 for l in lengths)
w = make_world('no communication')                                  # the planners choose from turn left, turn right, forward
assert list(w.envs.single_action_space.nvec) == [3, 3]
w.close()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()